# STAT 764 · Lab 9 — Explain it to the hospital

**Assigned Thu Nov 12 · Due Tue Nov 17, 11:59 pm · 25 points**

Individual work. About an hour, and the whole notebook runs in under a minute. You need Meeting 19's three
questions and their tools (permutation importance, partial dependence and ICE, SHAP values) and Meeting 20's
warnings about them.

Same rules as before: `git pull`, copy into `work/` before editing, Restart & Run All before submitting, upload
the `.ipynb` to Canvas. **This lab needs the `shap` package:** in your clone, run `git pull` and then `uv sync`,
and restart your kernel. If it will not install, Part 3 falls back to values saved for its three stays, and
nothing else needs it; tell me, so we can fix your setup.

**How it is graded.** Complete or incomplete, part by part. A part earns its points when everything it asks for is
in your notebook and has run. A written answer earns its points when it points at one of your numbers.

The hospital is thinking of using Meeting 17's boosted model to flag patients for the \$150 home visit. Before
anyone signs off, they want to know what the model has learned, in words a care manager can follow. You are
going to tell them: which columns it relies on, how its chance moves with one of them, why it scored three
particular patients as it did, and what they should not read into any of it.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:                       # you are inside (or just outside) your clone
    sys.path.insert(0, str(found[0] / "course"))
else:                           # Colab, or a copy saved outside the clone
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import expit
from stat764 import diabetes

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    import shap
except ImportError:
    shap = None
    print("shap is not installed, so Part 3 uses backup values saved for its three stays.\n"
          "To install it: in your clone run `git pull`, then `uv sync`, and restart the kernel (in Colab: !pip install shap).")

# The readmission data from Meetings 9-12, with Meeting 9's split by PATIENT (split 764).
enc = diabetes()
enc = enc[~enc["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])]  # death, hospice
y_all = (enc["readmitted"] == "<30").astype(int)
NUM = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["age", "gender", "race", "admission_type_id", "discharge_disposition_id",
       "admission_source_id", "A1Cresult", "max_glu_serum", "insulin", "change",
       "diabetesMed", "medical_specialty"]
X_all = enc[NUM + CAT].astype({**{c: str for c in CAT}, **{c: float for c in NUM}})
train, test = next(GroupShuffleSplit(1, test_size=0.25, random_state=764)
                   .split(X_all, y_all, groups=enc["patient_nbr"]))
X_test, y_test = X_all.iloc[test], y_all.iloc[test].to_numpy()

# Meeting 16's sample: a random 15,000 of the training patients, with all their stays.
pool = enc["patient_nbr"].iloc[train].unique()
chosen = np.random.default_rng(16).choice(pool, 15_000, replace=False)
rows = train[np.isin(enc["patient_nbr"].iloc[train], chosen)]
X_fit, y_fit = X_all.iloc[rows], y_all.iloc[rows].to_numpy()

prep = ColumnTransformer([
    ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT),
])
# Meeting 17's boosted model: 34 rounds at a learning rate of 0.1, trees of depth 3.
boost = Pipeline([("prep", prep), ("model", GradientBoostingClassifier(n_estimators=34, learning_rate=0.1,
                                                                        max_depth=3, random_state=0))]).fit(X_fit, y_fit)
p_boost = boost.predict_proba(X_test)[:, 1]
print(f"test: {len(y_test):,} stays · came back {y_test.mean():.1%} · boosting AUC {roc_auc_score(y_test, p_boost):.4f}")

In [ ]:
def curves(model, patients, column, values):
    """Each patient's predicted chance as `column` is set to each of `values`, everything else unchanged.

    One row per patient (an ICE curve), one column per value; the column means are the partial dependence.
    """
    out = []
    for v in values:
        changed = patients.copy()
        changed[column] = v
        out.append(model.predict_proba(changed)[:, 1])
    return np.array(out).T


def plot_curves(ice, values, xlabel):
    """Thin lines: 40 patients' ICE curves. Thick line: the partial dependence, the average of all of them."""
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(values, ice[:40].T, color="0.6", lw=0.8)
    ax.plot(values, ice.mean(axis=0), color="black", lw=3, label="partial dependence (the average)")
    ax.set(xlabel=xlabel, ylabel="predicted chance")
    ax.legend(loc="upper left")
    plt.show()


# Backup, used only if shap will not install: the boosted model's SHAP values for Part 3's three stays, computed
# with shap 0.54.1 on the instructor's computer (same data, seeds and settings), in log-odds.
BOOST_SHAP_BACKUP = {
    "base": -2.127137,
    5924: {"A1Cresult": 0.006032, "admission_source_id": -0.000703, "admission_type_id": 0.000568,
           "age": -0.082973, "change": 1.3e-05, "diabetesMed": 3e-05, "discharge_disposition_id": -0.083463,
           "gender": 0.00031, "insulin": -0.002372, "max_glu_serum": 0.0, "medical_specialty": -0.008186,
           "num_lab_procedures": 0.001302, "num_medications": 0.073999, "num_procedures": -0.005347,
           "number_diagnoses": 0.228125, "number_emergency": -0.010158, "number_inpatient": 2.714564,
           "number_outpatient": 0.001736, "race": 4.8e-05, "time_in_hospital": 0.007873},
    10997: {"A1Cresult": 0.004447, "admission_source_id": 0.00028, "admission_type_id": -0.000797,
            "age": -0.001458, "change": -2.6e-05, "diabetesMed": 0.000267, "discharge_disposition_id": -0.084411,
            "gender": 0.00031, "insulin": -0.003938, "max_glu_serum": 0.0, "medical_specialty": -0.008148,
            "num_lab_procedures": 0.003513, "num_medications": 0.073979, "num_procedures": 0.005092,
            "number_diagnoses": 0.218193, "number_emergency": 0.010334, "number_inpatient": 2.833522,
            "number_outpatient": 0.003624, "race": 4.8e-05, "time_in_hospital": -0.027164},
    21660: {"A1Cresult": 0.000335, "admission_source_id": 0.000241, "admission_type_id": -0.000897,
            "age": -0.006403, "change": -0.000274, "diabetesMed": 0.000474, "discharge_disposition_id": -0.213493,
            "gender": -0.000221, "insulin": -0.015255, "max_glu_serum": 0.0, "medical_specialty": -0.032286,
            "num_lab_procedures": -0.061034, "num_medications": -0.004459, "num_procedures": 0.001322,
            "number_diagnoses": -0.058959, "number_emergency": -0.003997, "number_inpatient": -0.239353,
            "number_outpatient": -0.001112, "race": 2.3e-05, "time_in_hospital": -0.046526},
}


def explain_tree(pipe, patients):
    """SHAP values of the boosted pipeline for some stays, from shap's TreeSHAP, summed by original column.

    Returns (table, base): one row per stay and one column per original column, in log-odds. base is the model's
    average log-odds over its training stays (about 10.6%); it is not Meeting 17's starting value, the 11.6% rate,
    because an average of log-odds comes out lower. base + a row's sum is that stay's log-odds, so expit() of it is the
    model's chance.
    Without shap it falls back to BOOST_SHAP_BACKUP, for Part 3's three stays only.
    """
    if shap is None:
        stays = list(X_test.index.get_indexer(patients.index))        # positions among the test stays
        missing = [i for i in stays if i not in BOOST_SHAP_BACKUP]
        if missing:
            raise RuntimeError(f"shap is not installed, and the saved values cover only Part 3's three stays, not positions "
                               f"{sorted({int(i) for i in missing})}. Check that a, b and c are positions among all the test "
                               "stays and that you passed X_test.iloc[[a, b, c]] with double brackets. To install shap: "
                               "git pull, then uv sync, and restart the kernel.")
        print("(shap is not installed: these values are the saved backup)")
        table = pd.DataFrame([BOOST_SHAP_BACKUP[i] for i in stays], index=patients.index)
        return table, BOOST_SHAP_BACKUP["base"]
    pre, model = pipe[:-1], pipe[-1]
    z = pre.transform(patients)
    z = z.toarray() if hasattr(z, "toarray") else np.asarray(z)
    explainer = shap.TreeExplainer(model)
    values = explainer.shap_values(z)
    names = pre.get_feature_names_out()                        # e.g. num__number_inpatient, cat__age_[80-90)
    column = [n.split("__", 1)[1] if n.startswith("num__") else
              max((c for c in CAT if n.split("__", 1)[1].startswith(c + "_")), key=len) for n in names]
    table = pd.DataFrame(values, columns=column, index=patients.index).T.groupby(level=0).sum().T
    return table, float(np.ravel(explainer.expected_value)[0])


ice_patients = X_test.sample(200, random_state=19)      # the same 200 test stays as Meeting 19
print("helpers ready: curves, plot_curves, explain_tree, ice_patients")

---

## Part 1 · Which columns does it rely on? (5 points)

**(i)** Permutation importance for `boost` twice, with `scoring="roc_auc"`, `n_repeats=5` and `random_state=764`:
once on the **test** stays (`X_test`, `y_test`) and once on the **training** stays (`X_fit`, `y_fit`).

**(ii)** One table: the 6 columns with the largest AUC drop on the test stays, with each one's drop on the
training stays beside it.

In [ ]:
# YOUR CODE HERE

## Part 2 · How does its chance move? (5 points)

Choose **one** of these count columns, and say which in a comment: `time_in_hospital`, `number_emergency`,
`number_outpatient` or `number_diagnoses`.

**(i)** Its ICE curves and partial dependence for `ice_patients`, using every whole number from the column's
smallest value in `X_test` up to its 99th percentile:
`values = np.arange(X_test[column].min(), X_test[column].quantile(0.99) + 1)`, then
`ice = curves(boost, ice_patients, column, values)` and `plot_curves(ice, values, column)`. Print the partial
dependence at each value: it is `ice.mean(axis=0)`.

**(ii)** How many **test** stays have each of those values? Print the counts beside the partial dependence, and
name the values with fewer than 100 stays (if there are none, say so, and name the value with the fewest).

In [ ]:
# YOUR CODE HERE

## Part 3 · Why these three patients? (5 points)

**(i)** Find three test stays, using `p_boost` and `y_test`:

- the stay with the **highest** chance among those who **came back**;
- the stay with the **highest** chance among those who **did not** come back;
- the stay with the **lowest** chance among those who **came back**.

`a`, `b`, `c` must be positions among **all** the test stays (0 to 24,705). `np.argmax(p_boost[y_test == 1])` is a
position among the came-back stays only; turn it back with `np.flatnonzero(y_test == 1)[...]`. Use `.iloc`, e.g.
`X_test["number_inpatient"].iloc[a]`. Print each one's chance, number of prior stays (`number_inpatient`) and
discharge type.

**(ii)** `explain_tree(boost, X_test.iloc[[a, b, c]])` for the three. A **push** is one column's SHAP value: how far
that column moves this stay's score from `base`, in log-odds (+ up, − down). `explain_tree` returns them unsorted, so
for each stay print its 4 largest pushes **by size, up or down**: `row.sort_values(key=abs, ascending=False).head(4)`.
Then check that `base` plus the row's sum, converted with `expit`, gives back its `p_boost`.

In [ ]:
# YOUR CODE HERE

## Part 4 · Written: the memo to the hospital (10 points)

Write it for a care manager: plain words, and one number from your output in each answer.

**(a)** (3 points) What does the model rely on? Name the columns and their importance. You computed it on test
stays and on training stays: which would you report to the hospital, and why?

**(b)** (2 points) How does the chance move with the column you chose in Part 2? Over which values does that
curve rest on many patients, and over which on few (or, if every value has 100 or more, the fewest)? A flat curve
is an answer too: say what it means.

**(c)** (3 points) The two high-risk stays from Part 3: one came back and one did not. Compare their
explanations. What does that tell you about what a SHAP explanation can and cannot say? Then: what, if anything,
did the model see in the low-risk stay who came back?

**(d)** (2 points) Two things the hospital should **not** read into these explanations, each tied to a different
one of Meeting 20's three questions: *Is something else carrying the same information?* (look-alike columns), *Were
there patients here?* (thin data), *Am I describing the model, or claiming a cause?* (not a cause).

### Your answers

*(double-click to edit)*

**(a)**

**(b)**

**(c)**

**(d)**

---

## Before you submit

- [ ] **Run → Restart Kernel and Run All Cells**
- [ ] Part 1 has both importance runs and the six-row table with the training drop beside the test drop
- [ ] Part 2 names your column, plots its curves, prints the partial dependence beside the counts, and names the
  values with fewer than 100 stays (or says there are none)
- [ ] Part 3 prints the three stays and their four largest pushes, with the check that they add up
- [ ] All four written answers are filled in, each pointing at a number
- [ ] Uploaded the `.ipynb` to Canvas

**25 points.**